In [27]:
import requests
import pandas as pd
import duckdb
from datetime import datetime

# -----------------------------
# API KEY
# -----------------------------
API_KEY = "f00d2bdb8d4e418a858163613262005"

# -----------------------------
# CONNECT TO DB
# -----------------------------
conn = duckdb.connect(
    "../database/airport_ops.duckdb"
)

# -----------------------------
# GET AIRPORTS
# -----------------------------
airports = conn.sql("""
SELECT
    iata_code,
    name,
    municipality,
    latitude_deg,
    longitude_deg
FROM airports
WHERE iata_code IS NOT NULL
""").df()

print(f"Loaded {len(airports)} airports")

# -----------------------------
# WEATHER LOOP
# -----------------------------
weather_rows = []

for _, airport in airports.iterrows():

    lat = airport["latitude_deg"]
    lon = airport["longitude_deg"]

    url = (
        "http://api.weatherapi.com"
        "/v1/current.json"
        f"?key={API_KEY}"
        f"&q={lat},{lon}"
    )

    try:

        response = requests.get(url)
        data = response.json()

        # skip bad responses
        if "current" not in data:
            print(
                f"Skipped "
                f"{airport['iata_code']}"
            )
            continue

        weather_rows.append({
            "airport_code":
                airport["iata_code"],

            "airport_name":
                airport["name"],

            "city":
                airport["municipality"],

            "temperature_c":
                data["current"]["temp_c"],

            "wind_kph":
                data["current"]["wind_kph"],

            "humidity":
                data["current"]["humidity"],

            "pressure_mb":
                data["current"]["pressure_mb"],

            "visibility_km":
                data["current"]["vis_km"],

            "weather_condition":
                data["current"]
                ["condition"]["text"],

            "last_updated":
                datetime.utcnow()
        })

    except Exception as e:
        print(e)

# -----------------------------
# CREATE DATAFRAME
# -----------------------------
weather = pd.DataFrame(
    weather_rows
)

print(weather.head())

print(
    f"Retrieved "
    f"{len(weather)} rows"
)

# -----------------------------
# SAVE TO DB
# -----------------------------
conn.register(
    "weather_df",
    weather
)

conn.execute("""
CREATE OR REPLACE TABLE weather AS
SELECT *
FROM weather_df
""")

print("Weather table created!")

conn.close()

Loaded 4563 airports


/tmp/ipykernel_16480/435931407.py:94: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.utcnow()


Skipped WFR
Skipped NKW
Skipped REA
Skipped CIS
Skipped MDY
Skipped AWK
  airport_code                        airport_name        city  temperature_c  \
0          OCA             Ocean Reef Club Airport   Key Largo           29.1   
1          WKK             Aleknagik / New Airport   Aleknagik            2.5   
2          BBQ  Burton-Nibbs International Airport  Codrington           26.6   
3          HIR       Honiara International Airport     Honiara           24.4   
4          MUA                       Munda Airport       Munda           26.6   

   wind_kph  humidity  pressure_mb  visibility_km   weather_condition  \
0      23.8        61       1018.0           10.0  Patchy rain nearby   
1       3.6        93       1009.0           10.0  Patchy rain nearby   
2      32.0        70       1017.0           10.0  Patchy rain nearby   
3       6.8        89       1008.0            9.0  Patchy rain nearby   
4       9.0        84       1008.0           10.0       Partly Cloudy   

  